# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [4]:
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [5]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [6]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [7]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [8]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

 # Limpieza del dataset orders

In [9]:
# verificamos duplicados con los indicadores unicos como id_pedido. Ya que un mismo usuario puede comprar varias veces.
print('Total de duplicados')
print(orders['id_pedido'].duplicated().sum())

Total de duplicados
100


In [10]:
# Los eliminamos ya que pueden afectar el analisis
orders = orders.drop_duplicates().reset_index(drop=True)

#verificamos que se hayan eliminado correctamente
print('Duplicados restantes:', orders.duplicated().sum())

Duplicados restantes: 0


In [11]:
# conversion de fecha en orders identificamos tipo de dato
orders['fecha_hora_pedido'].dtype

dtype('O')

In [12]:
#conversion de fecha en orders
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce', utc=True)

In [13]:
#verificamos que no haya errores
orders['fecha_hora_pedido'].isna().sum()

0

In [14]:
#confirmacion visual y confirmacion de tipo de dato
orders['fecha_hora_pedido'].head(5)

0   2025-05-22 00:00:00+00:00
1   2025-06-15 00:00:00+00:00
2   2025-05-02 00:00:00+00:00
3   2025-06-09 00:00:00+00:00
4   2025-03-30 00:00:00+00:00
Name: fecha_hora_pedido, dtype: datetime64[ns, UTC]

In [15]:
#Total de nulos por columna
columns = [['pais','dispositivo','fuente_referencia','nombre_producto','cantidad', 'precio_unitario','monto_descuento',]]
for col in columns:
    print('total_de_nulos:')
    print(orders[col].isna().sum())

total_de_nulos:
pais                 300
dispositivo           20
fuente_referencia     30
nombre_producto       30
cantidad              50
precio_unitario       50
monto_descuento       50
dtype: int64


In [16]:
# identificacion de valores unicos por columna para deteccion de nulos en columnas categoricas
# identificacion de posibles sentinels

object_columns = ['pais','dispositivo','fuente_referencia','nombre_producto', 'categoria_producto']

for col in object_columns:
    display(orders[col].unique())


array(['Argentina', 'Mexico', 'Colombia', 'mexico', 'colombia',
       'argentina', nan], dtype=object)

array(['desktop', 'mobile', nan], dtype=object)

array(['organic', 'paid_search', 'social', nan], dtype=object)

array(['Jacket-Winter-M', 'Tablet-Standard-64GB', 'Blender-XL-Red',
       'Laptop-Gaming-16GB', 'Sneakers-Urban-42', 'Phone-Pro-128GB',
       'Vacuum-Pro-Black', nan], dtype=object)

array(['Moda', 'Electronica', 'Hogar', nan], dtype=object)

In [17]:
#Multiples variables en Pais y categoria producto. Corregimos formatos
for col in object_columns:
    orders[col].str.strip().str.upper()

# Estandarizacion de pais y categoria_producto
orders['pais'] = (orders['pais'].str.strip()
    .str.replace('argentina', 'Argentina')
    .str.replace('mexico','Mexico')
    .str.replace('colombia', 'Colombia'))

orders['categoria_producto'] = orders['categoria_producto'].str.replace('Electronica', 'Electrónica')

# confirmamos cambios
object_columns = ['pais','dispositivo','fuente_referencia','nombre_producto', 'categoria_producto']
for col in object_columns:
    display(orders[col].unique())

array(['Argentina', 'Mexico', 'Colombia', nan], dtype=object)

array(['desktop', 'mobile', nan], dtype=object)

array(['organic', 'paid_search', 'social', nan], dtype=object)

array(['Jacket-Winter-M', 'Tablet-Standard-64GB', 'Blender-XL-Red',
       'Laptop-Gaming-16GB', 'Sneakers-Urban-42', 'Phone-Pro-128GB',
       'Vacuum-Pro-Black', nan], dtype=object)

array(['Moda', 'Electrónica', 'Hogar', nan], dtype=object)

In [18]:
# comprobacion de sentinels en columnas numericas.

num_columns = ['cantidad','precio_unitario','monto_descuento', 'monto_total']
print(orders[num_columns].describe())

           cantidad  precio_unitario  monto_descuento   monto_total
count  24950.000000     24950.000000     24950.000000  2.500000e+04
mean       7.115110       259.374527         4.502605  2.079498e+03
std      296.869961       138.690738         5.224057  9.914760e+04
min       -2.000000        20.030000         0.000000 -4.926500e+02
25%        1.000000       138.470000         0.000000  1.806700e+02
50%        2.000000       258.795000         0.000000  3.418050e+02
75%        2.000000       380.377500        10.000000  5.185800e+02
max    20000.000000       499.960000        15.000000  8.840200e+06


In [19]:
# negativos observados en cantidad y monto total, sentinel observado en cantidad
negativos = orders[(orders['cantidad'] < 0) | (orders['monto_total'] < 0)]
negativos

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13 00:00:00+00:00,NaN,desktop,paid_search,Phone-Pro-128GB,Electrónica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07 00:00:00+00:00,NaN,desktop,social,Phone-Pro-128GB,Electrónica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19 00:00:00+00:00,NaN,desktop,organic,Phone-Pro-128GB,Electrónica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25 00:00:00+00:00,NaN,desktop,paid_search,Phone-Pro-128GB,Electrónica,-1.0,423.53,0.0,-423.53


In [20]:
# Ya que las columnas estan completas, simplemente estan en negativo, lo convertimos en valor absoluto

orders['cantidad'] = orders['cantidad'].abs()
orders['monto_total'] = orders['monto_total'].abs()

In [21]:
#verificamos cambios en cantidad
orders[['cantidad','monto_total']].describe()

,cantidad,monto_total
count,24950.000000,2.500000e+04
mean,7.115511,2.079590e+03
std,296.869952,9.914760e+04
min,1.000000,5.240000e+00
25%,1.000000,1.807725e+02
50%,2.000000,3.419250e+02
75%,2.000000,5.185800e+02
max,20000.000000,8.840200e+06


In [22]:
# verificamos cuantas veces se encuentra este sentinel
orders['cantidad'].isin([20000]).sum()

4

In [23]:
# visualizacion de sentinels en la tabla
cantidad_sentinels = orders[(orders['cantidad'] == 20000)]
cantidad_sentinels

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3656,order_3656,user_884,2025-01-01 00:00:00+00:00,Argentina,mobile,organic,Laptop-Gaming-16GB,Electrónica,20000.0,297.66,0.0,5953200.0
3668,order_3668,user_7270,2025-06-24 00:00:00+00:00,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,20000.0,348.31,0.0,6966200.0
3722,order_3722,user_4723,2025-05-09 00:00:00+00:00,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electrónica,20000.0,442.01,0.0,8840200.0
3726,order_3726,user_2536,2025-02-12 00:00:00+00:00,Colombia,desktop,social,Laptop-Gaming-16GB,Electrónica,20000.0,290.85,0.0,5817000.0


In [24]:
# Solo se obserban 4 y juzgando por el monto total recibido, estas 4 incidencias aparentan ser outliers
# por lo mismo se conservaran para enriquecer el analisis.

In [25]:
# Porcentaje de nulos en orders
orders.isna().mean().sort_values(ascending=False)

pais                  0.0120
categoria_producto    0.0032
cantidad              0.0020
precio_unitario       0.0020
monto_descuento       0.0020
fuente_referencia     0.0012
nombre_producto       0.0012
dispositivo           0.0008
id_pedido             0.0000
id_usuario            0.0000
fecha_hora_pedido     0.0000
monto_total           0.0000
dtype: float64

In [26]:
# Se puede obeservar que en las columnas donde hay nulos todos se encuentran por debajo del 2% 
# por lo cual eliminarlos del dataset no representa una afectacion en el analisis

In [27]:
#confirmamos si podemos relacionar las columnas de pais y tipo de dispositivo para imputar nulos
missing_country_by_device = orders['pais'].isna().groupby(orders['dispositivo']).mean()
print(missing_country_by_device)

dispositivo
desktop    0.011486
mobile     0.012552
Name: pais, dtype: float64


In [28]:
#No parece haber algun tipo de relacion que nos permita asumir que algun pais depende estrictamente de un dispositivo.

In [29]:
# podriamos obtener un calculo para los valores faltantes entre las columnas de monto_total, cantidad, precio_unitario y monto_descuento
# Verficamos si podemos recuperar algunos de los nulos encontrados en cantidad

pd.crosstab(orders['cantidad'].isna(), orders['precio_unitario'].isna())

precio_unitario,False,True
cantidad,,
False,24950,0
True,0,50


In [30]:
# Esta tabla nos indica que los 50 datos nulos carecen de cantidad y precio unitario
# por lo cual es imposible sacar un calculo que podamos imputar.

In [31]:
# Verficamos si podemos recuperar algunos de los nulos encontrados en monto_total y monto_descuento

pd.crosstab(orders['monto_descuento'].isna(), orders['precio_unitario'].isna())

precio_unitario,False,True
monto_descuento,,
False,24950,0
True,0,50


In [32]:
#misma situacion entre el precio unitario y el monto descuento.

In [33]:
# Tomando en cuenta que no podemos imputar los valores nulos y que las columnas categoricas no dependen de otras columnas. 
# Procederemos a elimnar los nulos del dataset orders

In [34]:
# Eliminacion de nulos en orders
orders = orders.dropna()

In [35]:
# Verificamos que los cambios se han aplicado
orders.isna().sum()

id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64

# Limpieza de Dataset Marketing

In [36]:
# conversion de fecha para el data set marketing. 
marketing.info()

# flags observadas - columna fecha no esta con el formato correcto. 
# flags observadas - valores nulos en columna canal.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [37]:
#verificamos duplicados en marketing

print('Duplicados en marketing:', marketing.duplicated().sum())

Duplicados en marketing: 0


In [38]:
#verificacion de columnas categoricas
object_columns_m = ['pais','id_campaña','canal']
for col in object_columns_m:
    display(marketing[col].unique())

array(['Mexico', 'Colombia', 'Argentina'], dtype=object)

array(['organic_Mexico', 'paid_search_Mexico', 'social_Mexico',
       'organic_Colombia', 'paid_search_Colombia', 'social_Colombia',
       'organic_Argentina', 'paid_search_Argentina', 'social_Argentina'],
      dtype=object)

array(['organic', 'paid_search', 'social', nan], dtype=object)

In [39]:
#verificacion de columas numericas

print('Coulmna numerica')
marketing['gasto'].describe()

Coulmna numerica


count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64

In [40]:
# Al no haber nulos, podemos proceder a convertir a formato de fecha
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce', utc=True)

In [41]:
# confirmamos que el nuevo tipo de dato sea datetime64
marketing['fecha'].dtype

datetime64[ns, UTC]

In [42]:
# verificamos que no haya errores
marketing['fecha'].isna().sum()

0

In [43]:
#verificamos que se haya aplicado el cambio
marketing['fecha'].head()

0   2025-01-01 00:00:00+00:00
1   2025-01-01 00:00:00+00:00
2   2025-01-01 00:00:00+00:00
3   2025-01-01 00:00:00+00:00
4   2025-01-01 00:00:00+00:00
Name: fecha, dtype: datetime64[ns, UTC]

In [44]:
# limpieza de columna canal

# gracias a la exploracion previa notamos que canal y id_campaña estan relacionados.
print('columna canal')
marketing['canal'].value_counts()

columna canal


paid_search    507
social         506
organic        506
Name: canal, dtype: int64

In [45]:
print('columna id_campaña')
marketing['id_campaña'].value_counts()

columna id_campaña


organic_Argentina        180
paid_search_Mexico       180
social_Mexico            180
paid_search_Colombia     180
organic_Colombia         180
organic_Mexico           180
paid_search_Argentina    180
social_Colombia          180
social_Argentina         180
Name: id_campaña, dtype: int64

In [46]:
# Para facilitar la limpieza, extraeremos el canal diractamente del string de id_campaña
marketing['canal'] = marketing['canal'].fillna(
    marketing['id_campaña'].str.rsplit('_', n=1).str[0])


In [47]:
# confirmamos que se hayan aplicado los cambios a canal
marketing['canal'].isna().sum()

0

# Limpieza Catalog

In [48]:
# Verificamos que no haya sentinels en las columnas categoricas

object_columns_cat = ['nombre_producto','categoria_producto','proveedor']
for col in object_columns_cat:
    display(catalog[col].unique())

array(['Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Tablet-Standard-64GB',
       'Blender-XL-Red', 'Vacuum-Pro-Black', 'Sneakers-Urban-42',
       'Jacket-Winter-M'], dtype=object)

array(['Electrónica', 'Hogar', 'Moda'], dtype=object)

array(['Fuller, Pena and Myers', 'King Ltd', 'Bowers LLC', 'Long-Reid',
       'Rivera, Carr and Finley', 'Greene-Smith', 'Mcmillan-Rhodes'],
      dtype=object)

In [49]:
#Verificamos las columnas numericas
catalog['costo_unitario'].describe()

count      7.000000
mean     102.252857
std      111.011563
min       10.120000
25%       16.905000
50%       25.210000
75%      182.975000
max      280.680000
Name: costo_unitario, dtype: float64

In [50]:
# Revisamos que no haya duplicados en las columnas (nombre_producto + categoria_producto)
catalog.duplicated(subset=['nombre_producto', 'categoria_producto']).sum()

0

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [51]:
# exportar datasets

orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [52]:
# cargar archivos
orders_clean = pd.read_csv('orders_clean.csv')
catalog_clean = pd.read_csv('catalog_clean.csv')
marketing_clean = pd.read_csv('marketing_clean.csv')

In [ ]:
print(orders_clean.columns)
print()
print(marketing_clean.columns)
print()
print(catalog_clean.columns)

In [ ]:
# union de dataset orders_clean y marketing_clean para obtener revenue

orders_clean = orders_clean.rename(columns={'fuente_referencia': 'canal', 'fecha_hora_pedido': 'fecha' })
print(orders_clean.columns)

In [ ]:
# preparamos dataframes para join de tablas con pais como PK

orders_clean_corto = orders_clean[['id_usuario','id_pedido','fecha','pais','canal','cantidad','precio_unitario','monto_total']]
orders_clean_corto.head()

In [ ]:
# preparamos marketing tambien para el join
marketing_clean_corto = marketing_clean[['id_campaña','pais','fecha', 'canal', 'gasto']]
marketing_clean_corto.head()

In [ ]:
#comprobamos si podemos usar fecha y pais como PK en el join

print('fecha minima')
print(orders_clean_corto['fecha'].min())

print('fecha maxima')
print(orders_clean_corto['fecha'].max())


In [ ]:
print('fecha minima')
print(marketing_clean_corto['fecha'].min())
print('fecha maxima')
print(marketing_clean_corto['fecha'].max())

In [ ]:
# hacemos el merge
merged_left = pd.merge(orders_clean_corto, marketing_clean_corto, on=['pais','canal','fecha'], how='left') 

In [ ]:
# verificamos informacion del nuevo dataset despues del merge
merged_left.info()

In [ ]:
#exploramos la nueva tabla
merged_left.head()

In [ ]:
# verificamos duplicados
print('Duplicados despues del merge:', merged_left.duplicated().sum())

In [ ]:
#verificamos nulos despues del merge
merged_left.isna().sum()

In [ ]:
#eliminamos nulos
merged_left = merged_left.dropna()

#confirmamos eliminacion de nulos
merged_left.isna().sum()

In [ ]:
#Para calcular costos haremos un merge de orders_clean y catalog_clean
# Usaremos las columnas nombre_producto y categoria_producto para el join

# Verificamos que tengan valores iguales en nombre_producto
print('Valores en catalog_clean:', catalog_clean['nombre_producto'].unique())
print('Valores en orders_clean:', orders_clean['nombre_producto'].unique())

In [ ]:
# Verificamos que tengan valores iguales en categoria_producto

print('Valores en catalog_clean:', catalog_clean['categoria_producto'].unique())
print('Valores en orders_clean:', orders_clean['categoria_producto'].unique())

In [ ]:
#Al encontrar valores iguales en ambos datasets, podemos proceder al join 

#Creamos data sets nuevos a partir de la informacion que requerimos
catalog_clean_corto = catalog_clean[['nombre_producto','categoria_producto', 'costo_unitario']]
orders_clean_corto2 = orders_clean[['id_usuario','fecha','pais','nombre_producto','categoria_producto','cantidad','monto_total']]

# hacemos el join
cat_ord_left = pd.merge(orders_clean_corto2, catalog_clean_corto, on=['nombre_producto','categoria_producto'], how='left')

In [ ]:
# Verificamos el nuevo dataset
cat_ord_left.info()

In [ ]:
# No hay nulos, procedemos a explorar el nuevo set
cat_ord_left.head()

# Analisis de KPIs

In [ ]:
# Calculamos el ingreso total (revenue)
orders_clean['revenue'] = orders_clean['precio_unitario'] * orders_clean['cantidad']
print('Revenue total:', orders_clean['revenue'].sum().round(2))
print()

# Calculamos el costo
cat_ord_left['costo_total'] = cat_ord_left['costo_unitario'] * cat_ord_left['cantidad']
print('Costo total:', cat_ord_left['costo_total'].sum())
print()

# Calculamos el gasto total de Marketing
gasto_marketing = marketing['gasto'].sum()
print('Gasto total de Marketing', gasto_marketing.round(2))
print()

# Calculamos Profit
revenue_total = orders_clean['revenue'].sum().round(2)
costo_total = cat_ord_left['costo_total'].sum()
gasto_marketing_total = marketing['gasto'].sum()

profit = revenue_total - costo_total - gasto_marketing_total
print('Profit total:', round(profit, 2))
print()

# Calculamos ticket promedio por orden
ticket_promedio = merged_left.groupby('id_pedido')['monto_total'].sum().mean()
print('Ticket promedio por orden:', ticket_promedio.round(2))
print()

# Calculamos Cantidad promedio por orden
cantidad_promedio =  merged_left.groupby('id_pedido')['cantidad'].sum().mean()
print('Cantidad promedio por orden:', cantidad_promedio.round(2))


In [ ]:
# Para las ultimas dos preguntas graficamos para una mejor visualizacion

top_productos = cat_ord_left.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_productos.values, y=top_productos.index, palette='viridis')
plt.title('Productos más vendidos (por cantidad)')
plt.xlabel('Cantidad vendida')
plt.ylabel('Producto')
plt.tight_layout()
plt.show()

In [ ]:
# Graficamos el gasto por canal para tener una mejor visualizacion

gasto_por_canal = merged_left.groupby('canal')['gasto'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=gasto_por_canal.index, y=gasto_por_canal.values, palette='mako')
plt.title('Gasto de marketing por canal')
plt.xlabel('Canal')
plt.ylabel('Gasto total')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:
# Verficamos duplicados
# =========================
query_events = '''
SELECT id_usuario, nombre_evento, timestamp_evento, COUNT (*) AS numero_registros
FROM events
GROUP BY id_usuario, nombre_evento, timestamp_evento
ORDER by numero_registros DESC
;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

#parecen ser acciones genuinas, ya que un usuario puede entrar varias veces

In [ ]:
# Ya que hay duplicados agreagermos DISTINCT
# =========================
query_events = '''
SELECT DISTINCT id_usuario, nombre_evento, timestamp_evento
FROM events
GROUP BY id_usuario, nombre_evento, timestamp_evento
;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

#parecen ser acciones genuinas, ya que un usuario puede entrar varias veces

In [ ]:
# obtenemos rangos de fecha
# =========================
query_events = '''
SELECT MIN(timestamp_evento) AS min_time,
       MAX(timestamp_evento) AS max_time
FROM events;

'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:
# verificamos cuales son los steps del journey
# =========================
query_events = '''
SELECT distinct nombre_evento
FROM events;

'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:
# Construimos cortes por evento
# ======================

query_totals = '''
WITH first_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
select_item AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'select_item'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'

), 
add_to_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
begin_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
), 
add_payment_info AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
)
SELECT 
(SELECT COUNT(*) FROM first_visit) AS first_visit_users,
  (SELECT COUNT(*) FROM select_item) AS select_item_users,
  (SELECT COUNT(*) FROM add_to_cart) AS add_to_cart_users,
  (SELECT COUNT(*) FROM begin_checkout) AS begin_checkout_users,
  (SELECT COUNT(*) FROM add_payment_info) AS add_payment_info_users
;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# Construimos cortes por evento
# ======================

query_totals = '''
WITH first_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
select_item AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'select_item'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'

), 
add_to_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
begin_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
), 
add_payment_info AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
)
SELECT   
  ((SELECT COUNT(*) FROM first_visit) - (SELECT COUNT(*) FROM select_item)) * 100 
    / NULLIF((SELECT COUNT(*) FROM first_visit), 0) AS dropoff_after_first_visit_pct,
    
    ((SELECT COUNT(*) FROM select_item) - (SELECT COUNT(*) FROM add_to_cart)) * 100 
    / NULLIF((SELECT COUNT(*) FROM select_item), 0) AS dropoff_after_select_item_pct,
    
    ((SELECT COUNT(*) FROM add_to_cart) - (SELECT COUNT(*) FROM begin_checkout)) * 100 
    / NULLIF((SELECT COUNT(*) FROM add_to_cart), 0) AS dropoff_after_add_to_cart_pct,
    
    ((SELECT COUNT(*) FROM begin_checkout) - (SELECT COUNT(*) FROM add_payment_info)) * 100 
    / NULLIF((SELECT COUNT(*) FROM begin_checkout), 0) AS dropoff_after_begin_checkout_pct
  
;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

# Deteccion de puntos de abandono:
- first_visit con 2%
- add_to_cart con 5%
- begin_checkout con 13%

# Mayor dropoff en el funnel:
* El mayor dropoff se encuentra entre el paso 'add_to_cart' y 'begin_checkout'con una caida del 13%

In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH first_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
select_item AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'select_item'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'

), 
add_to_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
begin_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
), 
add_payment_info AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
)
SELECT 
ROUND((SELECT COUNT(*) FROM select_item) * 100.0 
    / NULLIF((SELECT COUNT(*) FROM first_visit), 0), 2) AS conversion_first_to_select_pct,
    
  ROUND((SELECT COUNT(*) FROM add_to_cart) * 100.0 
    / NULLIF((SELECT COUNT(*) FROM select_item), 0), 2) AS conversion_select_to_cart_pct,
    
  ROUND((SELECT COUNT(*) FROM begin_checkout) * 100.0 
    / NULLIF((SELECT COUNT(*) FROM add_to_cart), 0), 2) AS conversion_cart_to_checkout_pct,
    
  ROUND((SELECT COUNT(*) FROM add_payment_info) * 100.0 
    / NULLIF((SELECT COUNT(*) FROM begin_checkout), 0), 2) AS conversion_checkout_to_payment_pct
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head()

In [ ]:
# No hay usuarios con mas de 1 fecha de registro
# =========================
query_users = '''
SELECT id_usuario,
COUNT(fecha_registro) AS conteo_fecha
FROM users
GROUP BY id_usuario
ORDER BY conteo_fecha DESC
;
'''
users = pd.read_sql(query_users, con=engine)
users.head()

In [ ]:
# Armamos la corte por semana
# =========================
query_users = '''
SELECT id_usuario,
DATE_TRUNC('WEEK', CAST(MIN(fecha_registro) AS timestamp)) AS cohort_semanal
FROM users
GROUP BY id_usuario

;
'''
users = pd.read_sql(query_users, con=engine)
users.head()

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(5)

In [ ]:
# identificamos fechas
# =========================
query_user_activity = '''
SELECT MIN(fecha_actividad) AS Min_fecha,
       MAX(fecha_actividad) AS Max_fecha
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Conteo de usuarios
# =========================
query_user_activity = '''
SELECT
  ua.id_usuario,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 7 AND ua.activo = 1 THEN ua.id_usuario END) AS users_d7,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 14 AND ua.activo = 1 THEN ua.id_usuario END) AS users_d14,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 21 AND ua.activo = 1 THEN ua.id_usuario END) AS users_d21,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 28 AND ua.activo = 1 THEN ua.id_usuario END) AS users_d28
FROM user_activity AS ua
LEFT JOIN users AS u
  ON ua.id_usuario = u.id_usuario
WHERE ua.fecha_actividad BETWEEN '2025-01-08' AND '2025-06-28'
GROUP BY ua.id_usuario
ORDER BY ua.id_usuario
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(10)

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
  SELECT 
    id_usuario,
    DATE_TRUNC('week', CAST(fecha_registro AS timestamp)) AS semana_cohorte
  FROM users
  WHERE fecha_registro BETWEEN '2025-01-08' AND '2025-06-28'
),
tamano_cohorte AS (
  SELECT 
    semana_cohorte,
    COUNT(DISTINCT id_usuario) AS total_usuarios
  FROM cohortes
  GROUP BY semana_cohorte
),
actividad_cohorte AS (
  SELECT 
    c.semana_cohorte,
    a.id_usuario,
    (a.dias_despues_registro / 7) AS semana_periodo
  FROM cohortes c
  LEFT JOIN user_activity a
    ON c.id_usuario = a.id_usuario
  WHERE a.activo = 1
),
retencion AS (
  SELECT 
    semana_cohorte,
    semana_periodo,
    COUNT(DISTINCT id_usuario) AS usuarios_activos
  FROM actividad_cohorte
  GROUP BY semana_cohorte, semana_periodo
)
SELECT 
  r.semana_cohorte,
  r.semana_periodo,
  r.usuarios_activos,
  t.total_usuarios,
  ROUND(r.usuarios_activos * 100.0 / t.total_usuarios, 2) AS retencion_pct
FROM retencion r
JOIN tamano_cohorte t
  ON r.semana_cohorte = t.semana_cohorte
ORDER BY r.semana_cohorte, r.semana_periodo;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversion entre control y tratamiento se mantiene igual
   - **H₁ (Hipótesis alternativa):** La tasa de conversion entre control y tratamiento es distinta
   
**Test estadístico:** Analizaremos estas hipotesis con el uso de la Z-test 

**Nivel de significancia alpha:** 0.05

In [ ]:
# Cargamos el nuevo archivo
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

In [ ]:
# Exploramos el dataset
experiment.head()

In [ ]:
#Verificamos que no haya nulos
experiment.info()

In [ ]:
# deteccion de valores unicos en 'id_usuario'
print('Usuarios Únicos:', experiment['id_usuario'].nunique())

In [ ]:
# Resumen estadistico de columnas numericas
num_columns = experiment[['convirtio','duracion_sesion']]
for col in num_columns:
    print('Nombre de columna:', col)
    print(experiment[col].describe())

In [ ]:
# fecha maxima y minima para el rango del experimento
print("Fecha mínima:", experiment['timestamp'].min())
print("Fecha máxima:", experiment['timestamp'].max())

In [ ]:
# Explorar variables categóricas y cómo se distribuyen
columns = experiment[['variante','pais', 'dispositivo']]
print("\nConteo de categorías:")
for col in columns:
        print('Nombre de columna:', col)
        print(experiment[col].describe())

In [ ]:
# confirmamos que la muestra a utilizar sea relativamente similar en ambos grupos
experiment['variante'].value_counts()

In [ ]:
# verificamos que no haya sentinels en las columnas categoricas
for col in columns:
    display(experiment[col].unique())

In [ ]:
# Ya con una limpieza hecha procedemos a aplicar la Z-Test

# Numero de usuarios convertidos por experimento
conversiones = experiment.groupby('variante')['convirtio'].sum()

# total de usuarios por página
totales = experiment.groupby('variante')['convirtio'].count()

print('Usuarios convertidos por experimento:\n', conversiones)
print('Total de usuarios por experimento:\n', totales)

In [ ]:
# Pasamos valores obtenidos del total de convertidos a formato lista para aplicar la z-test
exitos = [conversiones['control'], conversiones['tratamiento']] 
print(exitos)

In [ ]:
# Pasamos valores obtenidos del total de usuarios a formato lista para aplicar la z-test
observaciones = [totales['control'], totales['tratamiento']]
print(observaciones)

In [ ]:
# Aplicar prueba
z_stat, p_value = proportions_ztest(exitos, observaciones)

# Visualizar resultados
print(f"Estadístico Z: {z_stat}")
print(f"Valor p: {p_value}")

In [ ]:
#Tasas de conversion

tasa_control = exitos[0] / observaciones[0]
tasa_tratamiento = exitos[1] / observaciones[1]

print(f'Tasa de conversión de grupo control: {tasa_control:.2%}')
print(f'Tasa de conversión de grupo tratamiento: {tasa_tratamiento:.2%}')

# Interpretar dirección de resultados
if tasa_control > tasa_tratamiento:
    print(f"\nEl grupo control tiene una mayor tasa de conversión ({tasa_control - tasa_tratamiento:.2%}).")
elif tasa_tratamiento > tasa_control:
    print(f"\nEl grupo tratamiento tiene una mayor tasa de conversión ({tasa_tratamiento - tasa_control:.2%})")
else:
    print("\nAmbos grupos tienen la misma tasa de conversión.")

In [ ]:
# Aplicamos la interpretacion de este calculo
alpha = 0.05 #umbral
if p_value < alpha:
    print('Hay evidencia estadística que rechaza la H0')
else:
    print('No hay evidencia estadística que rechace la H0')


### Interpretación

**Decisión:**  
Despues de correr el codigo y los calculos, podemos decir que no hay evidencia que rechace la hipotesis nula.

**Interpretación de negocio:**  
Se realizó un z-test para comparar las tasas de conversión entre el grupo "control" y "tratamiento". Con Z = -0.8132 y p = 0.41605, rechazamos H0 al nivel alpha = 0.05. No Hay evidencia estadística de que las tasas de conversión difieren entre ambos grupos. La tasa del grupo "tratamiento" fue [16.29%] vs [15.69%] del grupo "control". 
Dado que el grupo "tratamiento" presenta una mayor tasa de conversión y la diferencia no es estadísticamente significativa, se recomienda evaluar su implementación considerando costos y contexto de negocio.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [1]:
# Enlace de Dashboard
https://drive.google.com/drive/folders/1KIYSvckrDuLY8gDuElBf_l_ulZaAACMc?usp=sharing